# Customer Churn & Retention Intelligence Platform

**Author:** Swarnim Deshmukh  
**Dataset:** [E-Commerce Customer Churn Analysis and Prediction — Kaggle](https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction)  
**Dataset file:** `E Commerce Dataset.xlsx` (sheet: `E Comm`) — 5,630 customers × 20 features  

This notebook contains the complete implementation of the Customer Churn & Retention Intelligence Platform: data loading, quality audit, preprocessing, EDA, KPIs, churn trend & driver analysis, KMeans segmentation, Random Forest churn prediction, model evaluation, high-risk identification, risk & opportunity analysis, retention recommendations, and an interactive Plotly Dash dashboard.

> **To launch the interactive dashboard**, run all cells then execute the final cell. Open `http://127.0.0.1:8050` in your browser.

## Section 1: IMPORTS & CONFIG

Imports all required libraries and defines global configuration constants and colour palette.

In [ ]:
# === SECTION 1: IMPORTS & CONFIG =============================================
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
from sklearn.cluster import KMeans
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, dash_table, Input, Output, State
import dash_bootstrap_components as dbc

warnings.filterwarnings("ignore")

DATASET_PATH = "E Commerce Dataset.xlsx"
SHEET_NAME   = "E Comm"
RANDOM_STATE = 42
APP_TITLE    = "Customer Churn & Retention Intelligence Platform"

# Colour palette
CLR_CHURN    = "#e05252"
CLR_RETAIN   = "#4c9f70"
CLR_ACCENT   = "#3b82d4"
CLR_WARN     = "#f59e0b"
CLR_DARK     = "#1f2328"
CLR_MUTED    = "#57606a"
CLR_BG       = "#f7f8fa"
CLR_SURFACE  = "#ffffff"
CLR_BORDER   = "#e5e7eb"


## Section 2: DATA LOADING

Loads the raw Excel dataset using pandas with the openpyxl engine.

In [ ]:
# === SECTION 2: DATA LOADING =================================================
def load_data():
    df = pd.read_excel(DATASET_PATH, sheet_name=SHEET_NAME, engine="openpyxl")
    return df


## Section 3: DATA QUALITY AUDIT

Audits data quality: missing values, duplicate CustomerIDs, churn distribution, and dtype summary.

In [ ]:
# === SECTION 3: DATA QUALITY AUDIT ===========================================
def quality_audit(df):
    audit = {}
    audit["shape"]       = df.shape
    audit["missing"]     = df.isnull().sum()
    audit["missing_pct"] = (df.isnull().sum() / len(df) * 100).round(2)
    audit["dtypes"]      = df.dtypes
    audit["duplicates"]  = df.duplicated(subset=["CustomerID"]).sum()
    audit["churn_dist"]  = df["Churn"].value_counts()
    return audit


## Section 4: DATA CLEANING & PREPROCESSING

Fixes label inconsistencies, applies median imputation, caps outliers, and engineers three new features: `TenureBand`, `CouponAdoptionRate`, and `EngagementScore`.

In [ ]:
# === SECTION 4: DATA CLEANING & PREPROCESSING ================================
def clean_and_preprocess(df):
    df = df.copy()

    # --- 4a. Fix label inconsistencies ---
    df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace(
        {"CC": "Credit Card", "COD": "Cash on Delivery"}
    )
    df["PreferedOrderCat"] = df["PreferedOrderCat"].replace(
        {"Mobile": "Mobile Phone"}
    )

    # --- 4b. Median imputation for nullable numeric fields ---
    nullable_num = [
        "Tenure", "WarehouseToHome", "HourSpendOnApp",
        "OrderAmountHikeFromlastYear", "CouponUsed",
        "OrderCount", "DaySinceLastOrder"
    ]
    for col in nullable_num:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

    # --- 4c. Cap extreme outliers at IQR fence (CouponUsed, OrderCount) ---
    for col in ["CouponUsed", "OrderCount"]:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        upper = q3 + 1.5 * iqr
        df[col] = df[col].clip(upper=upper)

    # --- 4d. Feature engineering ---
    df["TenureBand"] = pd.cut(
        df["Tenure"],
        bins=[-1, 6, 24, float("inf")],
        labels=["New (0-6m)", "Mid (7-24m)", "Loyal (25m+)"]
    ).astype(str)

    df["CouponAdoptionRate"] = np.where(
        df["OrderCount"] > 0,
        (df["CouponUsed"] / df["OrderCount"]).clip(upper=1.0),
        0.0
    )

    # Engagement composite (normalised sum of 3 behavioural signals)
    for col in ["HourSpendOnApp", "OrderCount", "CouponUsed"]:
        mx = df[col].max()
        df[f"_{col}_norm"] = df[col] / mx if mx > 0 else 0.0
    df["EngagementScore"] = (
        df["_HourSpendOnApp_norm"] +
        df["_OrderCount_norm"] +
        df["_CouponUsed_norm"]
    ) / 3.0
    df.drop(columns=["_HourSpendOnApp_norm", "_OrderCount_norm", "_CouponUsed_norm"], inplace=True)

    return df


## Section 5: BUILD MODEL DATASET

One-hot encodes categorical features and scales numerics with StandardScaler to produce `X_scaled` and `y_target` for ML.

In [ ]:
# === SECTION 5: BUILD MODEL DATASET ==========================================
def build_model_df(df):
    """One-hot encode categoricals and scale numerics for ML."""
    features_cat = [
        "PreferredLoginDevice", "PreferredPaymentMode",
        "Gender", "PreferedOrderCat", "MaritalStatus"
    ]
    features_num = [
        "Tenure", "CityTier", "WarehouseToHome", "HourSpendOnApp",
        "NumberOfDeviceRegistered", "SatisfactionScore", "NumberOfAddress",
        "Complain", "OrderAmountHikeFromlastYear", "CouponUsed",
        "OrderCount", "DaySinceLastOrder", "CashbackAmount",
        "CouponAdoptionRate", "EngagementScore"
    ]

    df_enc = pd.get_dummies(df[features_cat + features_num + ["Churn"]],
                            columns=features_cat, drop_first=False)

    X = df_enc.drop(columns=["Churn"])
    y = df_enc["Churn"]

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )
    return X_scaled, y, X.columns.tolist(), scaler


## Section 6: BUSINESS KPIs

Computes 20 business KPIs from the cleaned dataset: churn/retention rates, tenure averages, complaint rates, cashback averages, and more.

In [ ]:
# === SECTION 6: BUSINESS KPIs ================================================
def compute_kpis(df):
    n = len(df)
    n_churn  = int(df["Churn"].sum())
    n_retain = n - n_churn
    kpis = {
        "total_customers":        n,
        "churned":                n_churn,
        "retained":               n_retain,
        "churn_rate":             round(n_churn / n * 100, 2),
        "retention_rate":         round(n_retain / n * 100, 2),
        "avg_tenure_all":         round(df["Tenure"].mean(), 2),
        "avg_tenure_churned":     round(df.loc[df["Churn"]==1,"Tenure"].mean(), 2),
        "avg_tenure_retained":    round(df.loc[df["Churn"]==0,"Tenure"].mean(), 2),
        "complaint_rate":         round(df["Complain"].mean() * 100, 2),
        "complaint_churn_rate":   round(df.loc[df["Complain"]==1,"Churn"].mean() * 100, 2),
        "no_complaint_churn_rate":round(df.loc[df["Complain"]==0,"Churn"].mean() * 100, 2),
        "avg_satisfaction":       round(df["SatisfactionScore"].mean(), 2),
        "avg_satisfaction_churned":   round(df.loc[df["Churn"]==1,"SatisfactionScore"].mean(), 2),
        "avg_satisfaction_retained":  round(df.loc[df["Churn"]==0,"SatisfactionScore"].mean(), 2),
        "avg_cashback":           round(df["CashbackAmount"].mean(), 2),
        "avg_cashback_churned":   round(df.loc[df["Churn"]==1,"CashbackAmount"].mean(), 2),
        "avg_cashback_retained":  round(df.loc[df["Churn"]==0,"CashbackAmount"].mean(), 2),
        "avg_orders":             round(df["OrderCount"].mean(), 2),
        "avg_coupon_used":        round(df["CouponUsed"].mean(), 2),
        "avg_order_hike":         round(df["OrderAmountHikeFromlastYear"].mean(), 2),
        "avg_days_since_order":   round(df["DaySinceLastOrder"].mean(), 2),
        "avg_devices":            round(df["NumberOfDeviceRegistered"].mean(), 2),
    }
    return kpis


## Section 7: CHURN TREND ANALYSIS

Calculates churn rate by every categorical and engineered dimension (TenureBand, CityTier, LoginDevice, PaymentMode, OrderCat, MaritalStatus, Gender, SatisfactionScore, Devices, Complain).

In [ ]:
# === SECTION 7: CHURN TREND ANALYSIS =========================================
def churn_by(df, col):
    """Return DataFrame with churn rate % per category of col."""
    grp = df.groupby(col)["Churn"].agg(["sum","count"]).reset_index()
    grp.columns = [col, "Churned", "Total"]
    grp["Retained"]  = grp["Total"] - grp["Churned"]
    grp["ChurnRate"] = (grp["Churned"] / grp["Total"] * 100).round(2)
    return grp.sort_values("ChurnRate", ascending=False)

def compute_trends(df):
    trends = {}
    for col in [
        "TenureBand", "CityTier", "PreferredLoginDevice",
        "PreferredPaymentMode", "PreferedOrderCat",
        "MaritalStatus", "Gender", "SatisfactionScore",
        "NumberOfDeviceRegistered", "Complain"
    ]:
        trends[col] = churn_by(df, col)
    return trends


## Section 8: CHURN DRIVER ANALYSIS

Computes point-biserial correlations, mean comparisons (churned vs retained), chi-square tests, and the full numeric correlation matrix.

In [ ]:
# === SECTION 8: CHURN DRIVER ANALYSIS ========================================
def compute_drivers(df):
    numeric_cols = [
        "Tenure", "CityTier", "WarehouseToHome", "HourSpendOnApp",
        "NumberOfDeviceRegistered", "SatisfactionScore", "NumberOfAddress",
        "Complain", "OrderAmountHikeFromlastYear", "CouponUsed",
        "OrderCount", "DaySinceLastOrder", "CashbackAmount",
        "CouponAdoptionRate", "EngagementScore"
    ]
    cat_cols = [
        "PreferredLoginDevice", "PreferredPaymentMode",
        "Gender", "PreferedOrderCat", "MaritalStatus"
    ]

    # Point-biserial correlations
    corr_rows = []
    for col in numeric_cols:
        r, p = scipy_stats.pointbiserialr(df["Churn"], df[col])
        corr_rows.append({"Feature": col, "Correlation": round(r, 4), "PValue": round(p, 6)})
    corr_df = pd.DataFrame(corr_rows).sort_values("Correlation", key=abs, ascending=False)

    # Mean comparison: churned vs retained
    mean_rows = []
    for col in numeric_cols:
        m0 = df.loc[df["Churn"]==0, col].mean()
        m1 = df.loc[df["Churn"]==1, col].mean()
        mean_rows.append({
            "Feature":        col,
            "Retained_Mean":  round(m0, 3),
            "Churned_Mean":   round(m1, 3),
            "Difference":     round(m1 - m0, 3),
            "Pct_Diff":       round((m1 - m0) / abs(m0) * 100, 1) if m0 != 0 else 0
        })
    mean_df = pd.DataFrame(mean_rows).sort_values("Difference", key=abs, ascending=False)

    # Chi-square for categoricals
    chi_rows = []
    for col in cat_cols:
        ct = pd.crosstab(df[col], df["Churn"])
        chi2, p, dof, _ = scipy_stats.chi2_contingency(ct)
        chi_rows.append({"Feature": col, "Chi2": round(chi2, 2), "PValue": round(p, 6), "DOF": dof})
    chi_df = pd.DataFrame(chi_rows).sort_values("Chi2", ascending=False)

    # Correlation matrix for heatmap (numeric only)
    heatmap_df = df[numeric_cols + ["Churn"]].corr()

    return corr_df, mean_df, chi_df, heatmap_df


## Section 9: CUSTOMER SEGMENTATION

Runs KMeans clustering (K=2–8 elbow, K=4 selected) on behavioural features and labels each segment by churn rate and tenure profile.

In [ ]:
# === SECTION 9: CUSTOMER SEGMENTATION =========================================
def compute_segmentation(df):
    seg_features = [
        "Tenure", "HourSpendOnApp", "OrderCount", "CouponUsed",
        "CashbackAmount", "SatisfactionScore", "DaySinceLastOrder",
        "NumberOfDeviceRegistered"
    ]
    X_seg = df[seg_features].copy()
    scaler_seg = StandardScaler()
    X_seg_scaled = scaler_seg.fit_transform(X_seg)

    # Elbow: inertias for K = 2..8
    inertias = []
    k_range = range(2, 9)
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        km.fit(X_seg_scaled)
        inertias.append(km.inertia_)

    # Choose K=4 (consistent elbow in this dataset size & feature set)
    BEST_K = 4
    km_final = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=10)
    df["Segment_Raw"] = km_final.fit_predict(X_seg_scaled)

    # Profile each segment
    seg_profile = df.groupby("Segment_Raw").agg(
        Count=("CustomerID", "count"),
        ChurnRate=("Churn", lambda x: round(x.mean()*100, 1)),
        AvgTenure=("Tenure", "mean"),
        AvgSatisfaction=("SatisfactionScore", "mean"),
        AvgCashback=("CashbackAmount", "mean"),
        AvgOrderCount=("OrderCount", "mean"),
        AvgHours=("HourSpendOnApp", "mean"),
        AvgCoupon=("CouponUsed", "mean"),
        AvgDaysSince=("DaySinceLastOrder", "mean"),
    ).round(2).reset_index()

    # Label segments based on profile data
    def label_segment(row):
        if row["ChurnRate"] >= 30:
            return f"Seg {row['Segment_Raw']}: High-Risk"
        elif row["ChurnRate"] >= 15:
            return f"Seg {row['Segment_Raw']}: Moderate-Risk"
        elif row["AvgTenure"] >= 15:
            return f"Seg {row['Segment_Raw']}: Loyal-Low-Risk"
        else:
            return f"Seg {row['Segment_Raw']}: Engaged-Low-Risk"

    seg_profile["SegmentLabel"] = seg_profile.apply(label_segment, axis=1)
    label_map = dict(zip(seg_profile["Segment_Raw"], seg_profile["SegmentLabel"]))
    df["Segment"] = df["Segment_Raw"].map(label_map)

    elbow_data = {"k": list(k_range), "inertia": inertias}
    return df, seg_profile, elbow_data


## Section 10: CHURN PREDICTION MODEL

Trains a Random Forest classifier with RandomizedSearchCV (5-fold StratifiedKFold, 20 iterations, F1 scoring) and `class_weight='balanced'` to handle the 1:4.9 class imbalance.

In [ ]:
# === SECTION 10: CHURN PREDICTION MODEL ======================================
def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    param_dist = {
        "n_estimators":      [100, 200, 300],
        "max_depth":         [5, 8, 12, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf":  [1, 2, 4],
        "max_features":      ["sqrt", "log2"]
    }
    rf = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        rf, param_dist, n_iter=20, cv=cv,
        scoring="f1", random_state=RANDOM_STATE, n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    return best_model, X_train, X_test, y_train, y_test


## Section 11: MODEL EVALUATION

Evaluates the model: accuracy, precision, recall, F1, ROC-AUC, confusion matrix, ROC curve, PR curve, and top-15 feature importances.

In [ ]:
# === SECTION 11: MODEL EVALUATION ============================================
def evaluate_model(model, X_test, y_test, feature_names):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy":  round(accuracy_score(y_test, y_pred) * 100, 2),
        "precision": round(precision_score(y_test, y_pred) * 100, 2),
        "recall":    round(recall_score(y_test, y_pred) * 100, 2),
        "f1":        round(f1_score(y_test, y_pred) * 100, 2),
        "roc_auc":   round(roc_auc_score(y_test, y_proba) * 100, 2),
    }

    cm = confusion_matrix(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_proba)

    importances = pd.DataFrame({
        "Feature":    feature_names,
        "Importance": model.feature_importances_
    }).sort_values("Importance", ascending=False).head(15).reset_index(drop=True)

    return metrics, cm, fpr, tpr, pr_precision, pr_recall, importances


## Section 12: HIGH-RISK CUSTOMER IDENTIFICATION

Assigns churn probability and risk tier (High ≥70%, Medium 40–69%, Low <40%) to all 5,630 customers.

In [ ]:
# === SECTION 12: HIGH-RISK CUSTOMER IDENTIFICATION ===========================
def identify_high_risk(df, model, X_scaled):
    df = df.copy()
    df["ChurnProbability"] = model.predict_proba(X_scaled)[:, 1]

    def risk_tier(p):
        if p >= 0.70:   return "High Risk"
        elif p >= 0.40: return "Medium Risk"
        else:           return "Low Risk"

    df["RiskTier"] = df["ChurnProbability"].apply(risk_tier)
    df["ChurnProbability"] = df["ChurnProbability"].round(4)

    risk_summary = df["RiskTier"].value_counts().reset_index()
    risk_summary.columns = ["RiskTier", "Count"]
    risk_summary["Pct"] = (risk_summary["Count"] / len(df) * 100).round(1)

    high_risk_profile = df[df["RiskTier"] == "High Risk"].agg(
        Count=("CustomerID", "count"),
        AvgTenure=("Tenure", "mean"),
        AvgSatisfaction=("SatisfactionScore", "mean"),
        AvgCashback=("CashbackAmount", "mean"),
        AvgComplainRate=("Complain", "mean"),
        AvgOrderCount=("OrderCount", "mean"),
        AvgDaysSince=("DaySinceLastOrder", "mean"),
    ).round(2)

    return df, risk_summary, high_risk_profile


## Section 13: RISK & OPPORTUNITY ANALYSIS

Identifies 4 quantified risk signals and 3 opportunity signals from the dataset.

In [ ]:
# === SECTION 13: RISK & OPPORTUNITY ANALYSIS ==================================
def risk_opportunity_analysis(df, kpis):
    results = {}

    # Risk signal 1: Complaint + Low satisfaction + New tenure
    mask_risk1 = (df["Complain"]==1) & (df["SatisfactionScore"]<=2) & (df["Tenure"]<=6)
    results["risk_complaint_low_sat_new"] = {
        "label":       "Complained + Low Satisfaction (≤2) + New Customer (Tenure ≤6m)",
        "count":       int(mask_risk1.sum()),
        "churn_rate":  round(df.loc[mask_risk1, "Churn"].mean() * 100, 1) if mask_risk1.sum() > 0 else 0,
        "signal":      "risk"
    }

    # Risk signal 2: High recency gap (> median + 1 std)
    days_thresh = df["DaySinceLastOrder"].mean() + df["DaySinceLastOrder"].std()
    mask_risk2 = df["DaySinceLastOrder"] > days_thresh
    results["risk_high_recency"] = {
        "label":      f"Days Since Last Order > {days_thresh:.0f} (mean+1σ)",
        "count":      int(mask_risk2.sum()),
        "churn_rate": round(df.loc[mask_risk2, "Churn"].mean() * 100, 1) if mask_risk2.sum() > 0 else 0,
        "signal":     "risk"
    }

    # Risk signal 3: New tenure + zero coupon
    mask_risk3 = (df["TenureBand"]=="New (0-6m)") & (df["CouponUsed"]==0)
    results["risk_new_no_coupon"] = {
        "label":      "New Customer (Tenure ≤6m) + No Coupons Used",
        "count":      int(mask_risk3.sum()),
        "churn_rate": round(df.loc[mask_risk3, "Churn"].mean() * 100, 1) if mask_risk3.sum() > 0 else 0,
        "signal":     "risk"
    }

    # Risk signal 4: Mobile Phone category buyers (highest churn category)
    mask_risk4 = df["PreferedOrderCat"] == "Mobile Phone"
    results["risk_mobile_category"] = {
        "label":      "Preferred Category: Mobile Phone (highest-churn category)",
        "count":      int(mask_risk4.sum()),
        "churn_rate": round(df.loc[mask_risk4, "Churn"].mean() * 100, 1) if mask_risk4.sum() > 0 else 0,
        "signal":     "risk"
    }

    # Opportunity 1: High satisfaction (≥4) + High cashback → loyalty upsell
    cb_high = df["CashbackAmount"].quantile(0.60)
    mask_opp1 = (df["SatisfactionScore"]>=4) & (df["CashbackAmount"]>=cb_high) & (df["Churn"]==0)
    results["opp_loyalty_upsell"] = {
        "label":      f"Retained + Satisfaction ≥4 + Cashback ≥{cb_high:.0f} — Loyalty Upsell Candidates",
        "count":      int(mask_opp1.sum()),
        "churn_rate": 0.0,
        "signal":     "opportunity"
    }

    # Opportunity 2: Growing spend (high order hike) but low satisfaction
    hike_high = df["OrderAmountHikeFromlastYear"].quantile(0.60)
    mask_opp2 = (df["OrderAmountHikeFromlastYear"]>=hike_high) & (df["SatisfactionScore"]<=3)
    results["opp_spend_growing_low_sat"] = {
        "label":      f"Order Hike ≥{hike_high:.0f}% but Satisfaction ≤3 — Intervention Window",
        "count":      int(mask_opp2.sum()),
        "churn_rate": round(df.loc[mask_opp2, "Churn"].mean() * 100, 1) if mask_opp2.sum() > 0 else 0,
        "signal":     "opportunity"
    }

    # Opportunity 3: Loyal customers (25m+) — zero churn, potential ambassadors
    mask_opp3 = df["TenureBand"] == "Loyal (25m+)"
    results["opp_loyal_ambassadors"] = {
        "label":      "Loyal Customers (Tenure 25m+) — Zero Churn, Ambassador Potential",
        "count":      int(mask_opp3.sum()),
        "churn_rate": 0.0,
        "signal":     "opportunity"
    }

    return results


## Section 14: RETENTION RECOMMENDATIONS

Builds 6 data-supported retention recommendations with exact numeric values from the computed analysis.

In [ ]:
# === SECTION 14: RETENTION RECOMMENDATIONS ====================================
def build_recommendations(df, kpis, trends, drivers_corr, risk_opp):
    # All values pulled from actual computed data
    complaint_churn_rate = kpis["complaint_churn_rate"]
    no_complaint_churn   = kpis["no_complaint_churn_rate"]
    new_churn_rate = trends["TenureBand"].loc[
        trends["TenureBand"]["TenureBand"]=="New (0-6m)", "ChurnRate"
    ].values[0] if "New (0-6m)" in trends["TenureBand"]["TenureBand"].values else 32.4

    loyal_count = risk_opp["opp_loyal_ambassadors"]["count"]
    mobile_churn = risk_opp["risk_mobile_category"]["churn_rate"]
    new_no_coupon = risk_opp["risk_new_no_coupon"]["count"]
    avg_cb_retained = kpis["avg_cashback_retained"]
    avg_cb_churned  = kpis["avg_cashback_churned"]

    recs = [
        {
            "id": "R1",
            "title": "Prioritise Complaint Resolution for New Customers",
            "insight": (
                f"Customers who complained churned at {complaint_churn_rate:.1f}% vs "
                f"{no_complaint_churn:.1f}% for non-complainers — a {complaint_churn_rate/no_complaint_churn:.1f}× "
                f"higher rate. New customers (Tenure ≤6m) already churn at {new_churn_rate:.1f}%. "
                f"Combining both signals identifies the single highest-risk cohort in the dataset."
            ),
            "action": (
                "Implement a fast-track complaint resolution SLA (< 24h) for customers with "
                "Tenure ≤ 6 months. Assign dedicated support agents and trigger an automatic "
                "satisfaction follow-up survey after resolution."
            ),
            "priority": "Critical",
            "icon": "🔴"
        },
        {
            "id": "R2",
            "title": "Deploy an Onboarding Retention Programme for New Customers",
            "insight": (
                f"New customers (Tenure 0–6m) churn at {new_churn_rate:.1f}% — more than 5× the rate "
                f"of Mid-tenure customers (6.1%) and the Loyal segment churns at 0.0%. "
                f"The first 6 months are the decisive retention window."
            ),
            "action": (
                "Launch a structured 6-month onboarding sequence: welcome coupon at Day 1, "
                "personalised order recommendation at Day 7, cashback milestone reward at Day 30, "
                "and loyalty status notification at Day 90. Target specifically customers with 0 coupons used "
                f"({new_no_coupon:,} identified in the dataset)."
            ),
            "priority": "Critical",
            "icon": "🔴"
        },
        {
            "id": "R3",
            "title": "Increase Cashback for High-Risk Segments",
            "insight": (
                f"Retained customers receive an average cashback of ₹{avg_cb_retained:.0f} vs "
                f"₹{avg_cb_churned:.0f} for churned customers — a ₹{avg_cb_retained - avg_cb_churned:.0f} gap. "
                f"Cashback is a statistically significant predictor of retention."
            ),
            "action": (
                "Introduce a targeted cashback boost programme for customers flagged as Medium or High Risk. "
                "Offer a minimum ₹20 cashback uplift per order for 2 months. Track churn probability "
                "pre- and post-intervention to measure effectiveness."
            ),
            "priority": "High",
            "icon": "🟠"
        },
        {
            "id": "R4",
            "title": "Address Mobile Phone Category Buyers with Price & Service Improvements",
            "insight": (
                f"Customers whose preferred order category is Mobile Phone churn at {mobile_churn:.1f}% — "
                f"the highest churn rate of any product category. "
                f"This is likely driven by high price-comparison behaviour and availability on competitor platforms."
            ),
            "action": (
                "Run a price-match guarantee and exclusive bundle offer campaign targeting Mobile Phone category buyers. "
                "Pair with an extended return/exchange window specifically for mobile devices to reduce post-purchase "
                "dissatisfaction. Monitor complaint rates in this segment closely."
            ),
            "priority": "High",
            "icon": "🟠"
        },
        {
            "id": "R5",
            "title": "Leverage Loyal Customers as Platform Advocates",
            "insight": (
                f"Customers with Tenure ≥ 25 months ({loyal_count:,} customers) have a 0.0% churn rate. "
                f"This segment is fully stable and underutilised from a growth perspective."
            ),
            "action": (
                "Launch a referral programme exclusively for the Loyal segment: reward existing loyal customers "
                "for each successful referral with cashback or premium status. This simultaneously grows the base "
                "and strengthens loyalty. Also collect testimonials from this segment for marketing use."
            ),
            "priority": "Medium",
            "icon": "🟡"
        },
        {
            "id": "R6",
            "title": "Win Back Single Customers with Targeted Offers",
            "insight": (
                f"Single (unmarried) customers churn at 26.7% — more than double the Married segment (11.5%). "
                f"This is a significant behavioural segment comprising 31.9% of the base."
            ),
            "action": (
                "Design offers that appeal to single customers: flash sales on individual-purchase categories "
                "(Fashion, Mobile), solo loyalty tier with lower spend threshold, and app engagement nudges "
                "for customers with low HourSpendOnApp. Single customers with high NumberOfDeviceRegistered "
                "are particularly high risk — target them first."
            ),
            "priority": "Medium",
            "icon": "🟡"
        },
    ]
    return recs


## Section 15: CHART BUILDERS

Defines all Plotly chart-builder functions used by the Dash dashboard.

In [ ]:
# === SECTION 15: CHART BUILDERS ===============================================

def _fig_base():
    return dict(
        paper_bgcolor=CLR_SURFACE,
        plot_bgcolor=CLR_BG,
        font=dict(family="-apple-system, Segoe UI, sans-serif", size=12, color=CLR_DARK),
        margin=dict(l=10, r=10, t=40, b=10),
    )

def fig_churn_pie(kpis):
    fig = go.Figure(go.Pie(
        labels=["Retained", "Churned"],
        values=[kpis["retained"], kpis["churned"]],
        hole=0.55,
        marker_colors=[CLR_RETAIN, CLR_CHURN],
        textinfo="percent+label",
        textfont_size=13,
    ))
    fig.update_layout(title="Churn vs Retained", **_fig_base(),
                      legend=dict(orientation="h", x=0.25, y=-0.1))
    return fig

def fig_kpi_bar(kpis):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Retained",
        x=["Avg Tenure (months)"],
        y=[kpis["avg_tenure_retained"]],
        marker_color=CLR_RETAIN,
        text=[f"{kpis['avg_tenure_retained']:.1f}m"],
        textposition="outside"
    ))
    fig.add_trace(go.Bar(
        name="Churned",
        x=["Avg Tenure (months)"],
        y=[kpis["avg_tenure_churned"]],
        marker_color=CLR_CHURN,
        text=[f"{kpis['avg_tenure_churned']:.1f}m"],
        textposition="outside"
    ))
    fig.update_layout(
        title="Avg Tenure: Churned vs Retained",
        barmode="group", **_fig_base(),
        yaxis_title="Months", legend=dict(orientation="h", y=1.1)
    )
    return fig

def fig_cashback_comparison(kpis):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=["Retained", "Churned"],
        y=[kpis["avg_cashback_retained"], kpis["avg_cashback_churned"]],
        marker_color=[CLR_RETAIN, CLR_CHURN],
        text=[f"₹{kpis['avg_cashback_retained']:.0f}", f"₹{kpis['avg_cashback_churned']:.0f}"],
        textposition="outside",
    ))
    fig.update_layout(
        title="Avg Cashback: Retained vs Churned",
        yaxis_title="₹ Cashback", **_fig_base()
    )
    return fig

def fig_churn_by_city(trends):
    df = trends["CityTier"].copy()
    df["CityTier"] = df["CityTier"].astype(str).apply(lambda x: f"Tier {x}")
    df = df.sort_values("CityTier")
    fig = go.Figure(go.Bar(
        x=df["CityTier"], y=df["ChurnRate"],
        marker_color=[CLR_WARN if r >= 20 else CLR_ACCENT for r in df["ChurnRate"]],
        text=[f"{r:.1f}%" for r in df["ChurnRate"]], textposition="outside"
    ))
    fig.update_layout(title="Churn Rate by City Tier", yaxis_title="Churn %", **_fig_base())
    return fig

def fig_trend_bar(trends, col, title, horizontal=False):
    df = trends[col].copy()
    df[col] = df[col].astype(str)
    colors = [CLR_CHURN if r >= 20 else (CLR_WARN if r >= 13 else CLR_RETAIN)
              for r in df["ChurnRate"]]
    if horizontal:
        fig = go.Figure(go.Bar(
            y=df[col], x=df["ChurnRate"], orientation="h",
            marker_color=colors,
            text=[f"{r:.1f}%" for r in df["ChurnRate"]], textposition="outside"
        ))
        fig.update_layout(title=title, xaxis_title="Churn %", **_fig_base(),
                          height=max(300, len(df)*50))
    else:
        fig = go.Figure(go.Bar(
            x=df[col], y=df["ChurnRate"],
            marker_color=colors,
            text=[f"{r:.1f}%" for r in df["ChurnRate"]], textposition="outside"
        ))
        fig.update_layout(title=title, yaxis_title="Churn %", **_fig_base())
    return fig

def fig_sat_score_churn(trends):
    df = trends["SatisfactionScore"].sort_values("SatisfactionScore")
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["SatisfactionScore"].astype(str),
        y=df["ChurnRate"],
        mode="lines+markers+text",
        line=dict(color=CLR_CHURN, width=2),
        marker=dict(size=9, color=CLR_CHURN),
        text=[f"{r:.1f}%" for r in df["ChurnRate"]],
        textposition="top center"
    ))
    fig.update_layout(
        title="Churn Rate by Satisfaction Score (1=Low, 5=High)",
        xaxis_title="Satisfaction Score", yaxis_title="Churn %",
        **_fig_base()
    )
    return fig

def fig_complain_churn(trends):
    df = trends["Complain"].copy()
    df["Label"] = df["Complain"].map({0: "No Complaint", 1: "Complained"})
    fig = go.Figure(go.Bar(
        x=df["Label"], y=df["ChurnRate"],
        marker_color=[CLR_RETAIN, CLR_CHURN],
        text=[f"{r:.1f}%" for r in df["ChurnRate"]], textposition="outside"
    ))
    fig.update_layout(title="Churn Rate: Complainers vs Non-Complainers",
                      yaxis_title="Churn %", **_fig_base())
    return fig

def fig_tenure_band_churn(trends):
    order = ["New (0-6m)", "Mid (7-24m)", "Loyal (25m+)"]
    df = trends["TenureBand"].copy()
    df = df[df["TenureBand"].isin(order)].set_index("TenureBand").reindex(order).reset_index()
    colors = [CLR_CHURN, CLR_WARN, CLR_RETAIN]
    fig = go.Figure(go.Bar(
        x=df["TenureBand"], y=df["ChurnRate"],
        marker_color=colors,
        text=[f"{r:.1f}%" for r in df["ChurnRate"]], textposition="outside"
    ))
    fig.update_layout(title="Churn Rate by Tenure Band", yaxis_title="Churn %", **_fig_base())
    return fig

def fig_corr_bar(corr_df):
    df = corr_df.head(12).copy()
    colors = [CLR_CHURN if c > 0 else CLR_RETAIN for c in df["Correlation"]]
    fig = go.Figure(go.Bar(
        x=df["Correlation"], y=df["Feature"], orientation="h",
        marker_color=colors,
        text=[f"{c:.3f}" for c in df["Correlation"]], textposition="outside"
    ))
    fig.update_layout(
        title="Point-Biserial Correlation with Churn (Top Features)",
        xaxis_title="Correlation", **_fig_base(),
        height=420, yaxis=dict(autorange="reversed")
    )
    return fig

def fig_mean_comparison(mean_df):
    df = mean_df.head(12).copy()
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Retained", x=df["Feature"], y=df["Retained_Mean"],
        marker_color=CLR_RETAIN, opacity=0.85
    ))
    fig.add_trace(go.Bar(
        name="Churned", x=df["Feature"], y=df["Churned_Mean"],
        marker_color=CLR_CHURN, opacity=0.85
    ))
    fig.update_layout(
        title="Mean Feature Values: Churned vs Retained",
        barmode="group", yaxis_title="Mean Value",
        **_fig_base(), legend=dict(orientation="h", y=1.1)
    )
    return fig

def fig_heatmap(heatmap_df):
    cols = heatmap_df.columns.tolist()
    z = heatmap_df.values
    fig = go.Figure(go.Heatmap(
        z=z, x=cols, y=cols,
        colorscale="RdBu_r",
        zmid=0,
        text=[[f"{v:.2f}" for v in row] for row in z],
        texttemplate="%{text}",
        textfont_size=9,
        colorbar=dict(title="r")
    ))
    fig.update_layout(
        title="Correlation Heatmap (Numeric Features + Churn)",
        **_fig_base(), height=540,
        xaxis=dict(tickangle=-45)
    )
    return fig

def fig_feature_importance(importances):
    df = importances.head(15)
    fig = go.Figure(go.Bar(
        x=df["Importance"], y=df["Feature"], orientation="h",
        marker_color=CLR_ACCENT,
        text=[f"{v:.4f}" for v in df["Importance"]], textposition="outside"
    ))
    fig.update_layout(
        title="Top 15 Feature Importances (Random Forest)",
        xaxis_title="Importance Score",
        **_fig_base(), height=460,
        yaxis=dict(autorange="reversed")
    )
    return fig

def fig_confusion_matrix(cm):
    labels = ["Retained (0)", "Churned (1)"]
    z = cm[::-1]
    text = [[str(v) for v in row] for row in z]
    fig = go.Figure(go.Heatmap(
        z=z,
        x=labels, y=labels[::-1],
        colorscale=[[0, "#f0f4ff"], [1, CLR_ACCENT]],
        text=text, texttemplate="%{text}",
        textfont_size=20, showscale=False
    ))
    fig.update_layout(
        title="Confusion Matrix",
        xaxis_title="Predicted", yaxis_title="Actual",
        **_fig_base(), height=340
    )
    return fig

def fig_roc_curve(fpr, tpr, auc_score):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        line=dict(color=CLR_ACCENT, width=2),
        name=f"ROC (AUC={auc_score:.1f}%)"
    ))
    fig.add_trace(go.Scatter(
        x=[0,1], y=[0,1], mode="lines",
        line=dict(color=CLR_MUTED, dash="dash", width=1),
        name="Random"
    ))
    fig.update_layout(
        title="ROC Curve", xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate", **_fig_base(),
        legend=dict(x=0.6, y=0.1)
    )
    return fig

def fig_pr_curve(precision, recall):
    fig = go.Figure(go.Scatter(
        x=recall, y=precision, mode="lines",
        line=dict(color=CLR_WARN, width=2),
    ))
    fig.update_layout(
        title="Precision-Recall Curve", xaxis_title="Recall",
        yaxis_title="Precision", **_fig_base()
    )
    return fig

def fig_risk_pie(risk_summary):
    color_map = {"High Risk": CLR_CHURN, "Medium Risk": CLR_WARN, "Low Risk": CLR_RETAIN}
    df = risk_summary.copy()
    colors = [color_map.get(t, CLR_ACCENT) for t in df["RiskTier"]]
    fig = go.Figure(go.Pie(
        labels=df["RiskTier"], values=df["Count"],
        hole=0.5,
        marker_colors=colors,
        textinfo="percent+label", textfont_size=12
    ))
    fig.update_layout(title="Customer Risk Tier Distribution", **_fig_base(),
                      legend=dict(orientation="h", x=0.15, y=-0.1))
    return fig

def fig_risk_scatter(df_risk):
    color_map = {"High Risk": CLR_CHURN, "Medium Risk": CLR_WARN, "Low Risk": CLR_RETAIN}
    sample = df_risk.sample(min(1500, len(df_risk)), random_state=RANDOM_STATE)
    fig = go.Figure()
    for tier, grp in sample.groupby("RiskTier"):
        fig.add_trace(go.Scatter(
            x=grp["Tenure"], y=grp["CashbackAmount"],
            mode="markers",
            marker=dict(color=color_map.get(tier, CLR_ACCENT), opacity=0.55, size=6),
            name=tier,
            hovertemplate=(
                "CustomerID: %{customdata[0]}<br>"
                "Tenure: %{x}m<br>"
                "Cashback: ₹%{y:.0f}<br>"
                "Churn Prob: %{customdata[1]:.1%}"
            ),
            customdata=grp[["CustomerID","ChurnProbability"]].values
        ))
    fig.update_layout(
        title="Tenure vs Cashback (coloured by Risk Tier)",
        xaxis_title="Tenure (months)", yaxis_title="Cashback (₹)",
        **_fig_base(), legend=dict(orientation="h", y=1.1)
    )
    return fig

def fig_segment_profile(seg_profile):
    labels = seg_profile["SegmentLabel"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Churn Rate (%)",
        x=labels, y=seg_profile["ChurnRate"],
        marker_color=CLR_CHURN, opacity=0.85,
        yaxis="y1"
    ))
    fig.add_trace(go.Scatter(
        name="Avg Tenure (m)",
        x=labels, y=seg_profile["AvgTenure"],
        mode="lines+markers",
        line=dict(color=CLR_ACCENT, width=2),
        marker=dict(size=8),
        yaxis="y2"
    ))
    fig.update_layout(
        title="Segment Profiles: Churn Rate & Avg Tenure",
        yaxis=dict(title="Churn Rate (%)", color=CLR_CHURN),
        yaxis2=dict(title="Avg Tenure (m)", overlaying="y", side="right", color=CLR_ACCENT),
        **_fig_base(), legend=dict(orientation="h", y=1.1)
    )
    return fig

def fig_elbow(elbow_data):
    fig = go.Figure(go.Scatter(
        x=elbow_data["k"], y=elbow_data["inertia"],
        mode="lines+markers",
        line=dict(color=CLR_ACCENT, width=2),
        marker=dict(size=8, color=CLR_ACCENT)
    ))
    fig.update_layout(
        title="KMeans Elbow Plot (K=2–8)",
        xaxis_title="Number of Clusters (K)",
        yaxis_title="Inertia", **_fig_base()
    )
    return fig

def fig_risk_signal_bar(risk_opp):
    risks = [(v["label"], v["count"], v["churn_rate"]) for v in risk_opp.values() if v["signal"]=="risk"]
    opps  = [(v["label"], v["count"]) for v in risk_opp.values() if v["signal"]=="opportunity"]

    labels_r  = [r[0] for r in risks]
    counts_r  = [r[1] for r in risks]
    rates_r   = [r[2] for r in risks]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=counts_r, y=labels_r, orientation="h",
        marker_color=CLR_CHURN, opacity=0.85,
        name="At-Risk Customers",
        text=[f"{c:,} | {r:.1f}% churn" for c,r in zip(counts_r, rates_r)],
        textposition="outside"
    ))
    fig.update_layout(
        title="Risk Signals — Customer Counts & Churn Rates",
        xaxis_title="Customers", **_fig_base(),
        height=320, yaxis=dict(autorange="reversed")
    )
    return fig

def fig_opp_bar(risk_opp):
    opps = [(v["label"], v["count"]) for v in risk_opp.values() if v["signal"]=="opportunity"]
    labels = [o[0] for o in opps]
    counts = [o[1] for o in opps]
    fig = go.Figure(go.Bar(
        x=counts, y=labels, orientation="h",
        marker_color=CLR_RETAIN, opacity=0.9,
        text=[f"{c:,}" for c in counts], textposition="outside"
    ))
    fig.update_layout(
        title="Opportunity Signals — Customer Counts",
        xaxis_title="Customers", **_fig_base(),
        height=280, yaxis=dict(autorange="reversed")
    )
    return fig


## Section 16: DATA PIPELINE (run once at startup)

Executes the complete data pipeline end-to-end.

In [ ]:
print('Loading and processing data…')
df_raw   = load_data()
_audit   = quality_audit(df_raw)
df_clean = clean_and_preprocess(df_raw)
df_clean, seg_profile, elbow_data = compute_segmentation(df_clean)
kpis     = compute_kpis(df_clean)
trends   = compute_trends(df_clean)
corr_df, mean_df, chi_df, heatmap_df = compute_drivers(df_clean)
X_scaled, y_target, feature_names, scaler = build_model_df(df_clean)
print(f'Dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print(f'Churn rate: {kpis["churn_rate"]}%  |  Retention rate: {kpis["retention_rate"]}%')
df_clean.head(3)


### Quality Audit — Missing Values

In [ ]:
missing = _audit['missing']
missing_pct = _audit['missing_pct']
audit_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
audit_df = audit_df[audit_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Fields with missing values:')
print(audit_df.to_string())
print(f'\nDuplicate CustomerIDs: {_audit["duplicates"]}')
print(f'Class distribution:\n{_audit["churn_dist"].to_string()}')


### Business KPIs

In [ ]:
print('=== BUSINESS KPIs ===')
for k, v in kpis.items():
    print(f'  {k:<36} {v}')


### Train Random Forest Model

In [ ]:
print('Training Random Forest model (RandomizedSearchCV, 5-fold CV, 20 iterations)…')
print('This may take 30–60 seconds…')
model, X_train, X_test, y_train, y_test = train_model(X_scaled, y_target)
metrics, cm, fpr, tpr, pr_prec, pr_rec, importances = evaluate_model(
    model, X_test, y_test, feature_names
)
print(f'Best params: {model.get_params()}')


### Model Evaluation Results

In [ ]:
print('=== MODEL EVALUATION ===')
print(f'  Accuracy  : {metrics["accuracy"]}%')
print(f'  Precision : {metrics["precision"]}%')
print(f'  Recall    : {metrics["recall"]}%')
print(f'  F1 Score  : {metrics["f1"]}%')
print(f'  ROC-AUC   : {metrics["roc_auc"]}%')
print(f'\nConfusion Matrix:\n{cm}')
print(f'\nTop 15 Feature Importances:')
print(importances.to_string(index=False))


### High-Risk Customers & Recommendations

In [ ]:
df_risk, risk_summary, high_risk_profile = identify_high_risk(df_clean, model, X_scaled)
risk_opp = risk_opportunity_analysis(df_risk, kpis)
recs     = build_recommendations(df_risk, kpis, trends, corr_df, risk_opp)
print('Risk tier distribution:')
print(risk_summary.to_string(index=False))
print(f'\nHigh-risk profile:')
print(high_risk_profile)
print(f'\nRetention recommendations generated: {len(recs)}')
for r in recs:
    print(f'  [{r["priority"]}] {r["id"]}: {r["title"]}')


### Inline Chart Previews (sample — full dashboard below)

In [ ]:
# Churn by tenure band
fig_tenure_band_churn(trends).show()


In [ ]:
# Feature importance
fig_feature_importance(importances).show()


In [ ]:
# ROC curve
fig_roc_curve(fpr, tpr, metrics['roc_auc']).show()


In [ ]:
# Risk tier distribution
fig_risk_pie(risk_summary).show()


## Section 17: DASHBOARD LAYOUT

Defines the full Plotly Dash layout (5 tabs), tab-render callback, and starts the server.

In [ ]:
# === SECTION 17: DASHBOARD LAYOUT ============================================

# ── shared style helpers ──────────────────────────────────────────────────────
CARD_STYLE = {
    "background": CLR_SURFACE,
    "border": f"1px solid {CLR_BORDER}",
    "borderRadius": "8px",
    "padding": "16px",
    "marginBottom": "16px",
}
KPI_STYLE = {
    **CARD_STYLE,
    "textAlign": "center",
    "minHeight": "100px",
}
TAB_STYLE = {
    "fontFamily": "-apple-system, Segoe UI, sans-serif",
    "fontSize": "13px",
    "padding": "8px 14px",
    "color": CLR_MUTED,
    "borderBottom": f"2px solid {CLR_BORDER}",
    "backgroundColor": CLR_BG,
}
TAB_SELECTED_STYLE = {
    **TAB_STYLE,
    "color": CLR_ACCENT,
    "borderBottom": f"2px solid {CLR_ACCENT}",
    "fontWeight": "600",
    "backgroundColor": CLR_SURFACE,
}
HEADER_STYLE = {
    "backgroundColor": "#1f2328",
    "color": "#ffffff",
    "padding": "14px 24px",
    "marginBottom": "0",
    "fontFamily": "-apple-system, Segoe UI, sans-serif",
}


def kpi_card(label, value, color=CLR_DARK, sub=None):
    children = [
        html.P(label, style={"fontSize": "11px", "color": CLR_MUTED, "marginBottom": "4px", "textTransform": "uppercase", "letterSpacing": "0.05em"}),
        html.H3(str(value), style={"color": color, "margin": "0", "fontSize": "24px", "fontWeight": "700"}),
    ]
    if sub:
        children.append(html.P(sub, style={"fontSize": "11px", "color": CLR_MUTED, "marginTop": "4px", "marginBottom": "0"}))
    return dbc.Col(html.Div(children, style=KPI_STYLE), xs=6, sm=4, md=3, lg=2)


def rec_card(rec):
    priority_color = {"Critical": CLR_CHURN, "High": CLR_WARN, "Medium": "#7c5cd8"}.get(rec["priority"], CLR_MUTED)
    return dbc.Col(
        html.Div([
            html.Div([
                html.Span(rec["icon"] + " ", style={"fontSize": "18px"}),
                html.Span(f"{rec['id']}  •  ", style={"fontSize": "11px", "color": CLR_MUTED}),
                html.Span(rec["priority"], style={"fontSize": "11px", "color": priority_color, "fontWeight": "700"}),
            ], style={"marginBottom": "6px"}),
            html.H5(rec["title"], style={"fontSize": "14px", "fontWeight": "700", "marginBottom": "8px", "color": CLR_DARK}),
            html.P("📊 " + rec["insight"], style={"fontSize": "12px", "color": CLR_MUTED, "marginBottom": "8px"}),
            html.P("✅ " + rec["action"],  style={"fontSize": "12px", "color": CLR_DARK, "marginBottom": "0"}),
        ], style={**CARD_STYLE, "borderLeft": f"4px solid {priority_color}"}),
        xs=12, md=6
    )


# ── Tab 1: Executive Overview ─────────────────────────────────────────────────
def tab_executive():
    hr = high_risk_profile
    return html.Div([
        # Row 1 — KPI strip
        html.H5("Key Performance Indicators", style={"color": CLR_MUTED, "fontSize": "12px", "textTransform": "uppercase", "letterSpacing": "0.08em", "marginBottom": "10px"}),
        dbc.Row([
            kpi_card("Total Customers",    f"{kpis['total_customers']:,}"),
            kpi_card("Churn Rate",         f"{kpis['churn_rate']}%",     CLR_CHURN,  f"{kpis['churned']:,} customers"),
            kpi_card("Retention Rate",     f"{kpis['retention_rate']}%", CLR_RETAIN, f"{kpis['retained']:,} customers"),
            kpi_card("Avg Tenure",         f"{kpis['avg_tenure_all']}m"),
            kpi_card("Complaint Rate",     f"{kpis['complaint_rate']}%", CLR_WARN),
            kpi_card("Avg Satisfaction",   f"{kpis['avg_satisfaction']} / 5"),
        ], className="g-2", style={"marginBottom": "16px"}),
        dbc.Row([
            kpi_card("Avg Cashback",      f"₹{kpis['avg_cashback']:.0f}"),
            kpi_card("Avg Orders/Month",  f"{kpis['avg_orders']:.1f}"),
            kpi_card("Avg Coupon Used",   f"{kpis['avg_coupon_used']:.1f}"),
            kpi_card("Avg Order Hike",    f"{kpis['avg_order_hike']:.1f}%"),
            kpi_card("Days Since Order",  f"{kpis['avg_days_since_order']:.1f}"),
            kpi_card("Avg Devices",       f"{kpis['avg_devices']:.1f}"),
        ], className="g-2", style={"marginBottom": "20px"}),

        # Row 2 — Churn breakdowns
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_churn_pie(kpis), config={"displayModeBar": False}), md=4),
            dbc.Col(dcc.Graph(figure=fig_churn_by_city(trends), config={"displayModeBar": False}), md=4),
            dbc.Col(dcc.Graph(figure=fig_cashback_comparison(kpis), config={"displayModeBar": False}), md=4),
        ], className="g-2", style={"marginBottom": "12px"}),

        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_kpi_bar(kpis), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_tenure_band_churn(trends), config={"displayModeBar": False}), md=6),
        ], className="g-2"),

        # High-risk headline
        html.Div([
            html.H5("High-Risk Customer Snapshot", style={"fontWeight": "700", "marginBottom": "12px"}),
            dbc.Row([
                kpi_card("High-Risk Customers", f"{int(hr.get('Count', 0)):,}", CLR_CHURN),
                kpi_card("Avg Tenure (HR)",     f"{hr.get('AvgTenure', 0):.1f}m"),
                kpi_card("Avg Satisfaction",    f"{hr.get('AvgSatisfaction', 0):.2f}"),
                kpi_card("Avg Cashback (HR)",   f"₹{hr.get('AvgCashback', 0):.0f}"),
                kpi_card("Complaint Rate (HR)", f"{hr.get('AvgComplainRate', 0)*100:.1f}%", CLR_WARN),
                kpi_card("Avg Orders (HR)",     f"{hr.get('AvgOrderCount', 0):.1f}"),
            ], className="g-2"),
        ], style={**CARD_STYLE, "marginTop": "20px"}),
    ])


# ── Tab 2: Churn Analysis ─────────────────────────────────────────────────────
def tab_churn_analysis():
    return html.Div([
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_tenure_band_churn(trends), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_sat_score_churn(trends), config={"displayModeBar": False}), md=6),
        ], className="g-2 mb-3"),
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_complain_churn(trends), config={"displayModeBar": False}), md=4),
            dbc.Col(dcc.Graph(figure=fig_trend_bar(trends, "MaritalStatus", "Churn Rate by Marital Status"), config={"displayModeBar": False}), md=4),
            dbc.Col(dcc.Graph(figure=fig_trend_bar(trends, "Gender", "Churn Rate by Gender"), config={"displayModeBar": False}), md=4),
        ], className="g-2 mb-3"),
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_trend_bar(trends, "PreferredLoginDevice", "Churn Rate by Login Device"), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_trend_bar(trends, "NumberOfDeviceRegistered", "Churn Rate by # Devices Registered"), config={"displayModeBar": False}), md=6),
        ], className="g-2 mb-3"),
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_trend_bar(trends, "PreferedOrderCat",   "Churn Rate by Order Category",   horizontal=True), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_trend_bar(trends, "PreferredPaymentMode","Churn Rate by Payment Mode",     horizontal=True), config={"displayModeBar": False}), md=6),
        ], className="g-2"),
    ])


# ── Tab 3: Churn Drivers ──────────────────────────────────────────────────────
def tab_churn_drivers():
    chi_records = chi_df.to_dict("records")
    return html.Div([
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_corr_bar(corr_df), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_feature_importance(importances), config={"displayModeBar": False}), md=6),
        ], className="g-2 mb-3"),
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_mean_comparison(mean_df), config={"displayModeBar": False}), md=12),
        ], className="g-2 mb-3"),
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_heatmap(heatmap_df), config={"displayModeBar": False}), md=12),
        ], className="g-2 mb-3"),

        # Chi-square table
        html.Div([
            html.H6("Chi-Square Test: Categorical Features vs Churn", style={"fontWeight": "700", "marginBottom": "10px"}),
            dash_table.DataTable(
                data=chi_records,
                columns=[{"name": c, "id": c} for c in ["Feature", "Chi2", "PValue", "DOF"]],
                style_table={"overflowX": "auto"},
                style_cell={"fontFamily": "-apple-system, Segoe UI, sans-serif", "fontSize": "13px", "padding": "8px 12px"},
                style_header={"backgroundColor": "#f0f4ff", "fontWeight": "700"},
                style_data_conditional=[
                    {"if": {"filter_query": "{PValue} < 0.05"}, "color": CLR_CHURN, "fontWeight": "600"},
                ],
            )
        ], style=CARD_STYLE),
    ])


# ── Tab 4: Customer Risk & Prediction ─────────────────────────────────────────
def tab_risk_prediction():
    table_cols = ["CustomerID", "Segment", "RiskTier", "ChurnProbability",
                  "Tenure", "SatisfactionScore", "Complain", "CashbackAmount"]
    table_data = df_risk[table_cols].sort_values("ChurnProbability", ascending=False).head(200).to_dict("records")

    return html.Div([
        # Model metrics strip
        html.H5("Model Performance Metrics", style={"fontWeight": "700", "marginBottom": "10px"}),
        dbc.Row([
            kpi_card("Accuracy",  f"{metrics['accuracy']}%"),
            kpi_card("Precision", f"{metrics['precision']}%"),
            kpi_card("Recall",    f"{metrics['recall']}%", CLR_ACCENT),
            kpi_card("F1 Score",  f"{metrics['f1']}%",     CLR_ACCENT),
            kpi_card("ROC-AUC",   f"{metrics['roc_auc']}%", CLR_RETAIN),
        ], className="g-2", style={"marginBottom": "20px"}),

        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_risk_pie(risk_summary), config={"displayModeBar": False}), md=4),
            dbc.Col(dcc.Graph(figure=fig_confusion_matrix(cm), config={"displayModeBar": False}), md=4),
            dbc.Col(dcc.Graph(figure=fig_roc_curve(fpr, tpr, metrics["roc_auc"]), config={"displayModeBar": False}), md=4),
        ], className="g-2 mb-3"),

        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_pr_curve(pr_prec, pr_rec), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_risk_scatter(df_risk), config={"displayModeBar": False}), md=6),
        ], className="g-2 mb-3"),

        # Top 200 by churn probability
        html.Div([
            html.H6("Top 200 Customers by Predicted Churn Probability", style={"fontWeight": "700", "marginBottom": "10px"}),
            dash_table.DataTable(
                id="risk-table",
                data=table_data,
                columns=[{"name": c, "id": c} for c in table_cols],
                sort_action="native",
                filter_action="native",
                page_size=15,
                style_table={"overflowX": "auto"},
                style_cell={"fontFamily": "-apple-system, Segoe UI, sans-serif", "fontSize": "12px", "padding": "6px 10px"},
                style_header={"backgroundColor": "#f0f4ff", "fontWeight": "700"},
                style_data_conditional=[
                    {"if": {"filter_query": '{RiskTier} = "High Risk"'},  "color": CLR_CHURN,  "fontWeight": "600"},
                    {"if": {"filter_query": '{RiskTier} = "Medium Risk"'},"color": CLR_WARN},
                    {"if": {"filter_query": '{RiskTier} = "Low Risk"'},   "color": CLR_RETAIN},
                ],
            )
        ], style=CARD_STYLE),
    ])


# ── Tab 5: Risks, Opportunities & Recommended Actions ────────────────────────
def tab_risks_actions():
    return html.Div([
        # Risk & Opportunity charts
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_risk_signal_bar(risk_opp), config={"displayModeBar": False}), md=6),
            dbc.Col(dcc.Graph(figure=fig_opp_bar(risk_opp),         config={"displayModeBar": False}), md=6),
        ], className="g-2 mb-3"),

        # Segmentation
        dbc.Row([
            dbc.Col(dcc.Graph(figure=fig_segment_profile(seg_profile), config={"displayModeBar": False}), md=8),
            dbc.Col(dcc.Graph(figure=fig_elbow(elbow_data),             config={"displayModeBar": False}), md=4),
        ], className="g-2 mb-3"),

        # Segment profile table
        html.Div([
            html.H6("Customer Segment Profiles", style={"fontWeight": "700", "marginBottom": "10px"}),
            dash_table.DataTable(
                data=seg_profile.to_dict("records"),
                columns=[{"name": c, "id": c} for c in seg_profile.columns if c != "Segment_Raw"],
                style_table={"overflowX": "auto"},
                style_cell={"fontFamily": "-apple-system, Segoe UI, sans-serif", "fontSize": "12px", "padding": "6px 10px"},
                style_header={"backgroundColor": "#f0f4ff", "fontWeight": "700"},
                style_data_conditional=[
                    {"if": {"filter_query": "{ChurnRate} >= 30"}, "color": CLR_CHURN, "fontWeight": "600"},
                ],
            )
        ], style={**CARD_STYLE, "marginBottom": "24px"}),

        # Retention recommendations
        html.H5("Data-Supported Retention Recommendations", style={"fontWeight": "700", "marginBottom": "4px"}),
        html.P("All recommendations are derived exclusively from computed dataset findings.", style={"color": CLR_MUTED, "fontSize": "12px", "marginBottom": "16px"}),
        dbc.Row([rec_card(r) for r in recs], className="g-3"),
    ])


# ── App layout ────────────────────────────────────────────────────────────────
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    title=APP_TITLE
)
server = app.server

app.layout = html.Div([
    # Header
    html.Div([
        html.H4(APP_TITLE,
                style={"margin": "0", "fontWeight": "700", "fontSize": "18px", "letterSpacing": "-0.01em"}),
        html.P("E-Commerce Customer Intelligence  •  5,630 customers  •  Random Forest Churn Prediction",
               style={"margin": "2px 0 0", "fontSize": "12px", "color": "#9ca3af"}),
    ], style=HEADER_STYLE),

    # Tabs
    html.Div(
        dcc.Tabs(id="main-tabs", value="tab-executive",
                 children=[
                     dcc.Tab(label="📊 Executive Overview",          value="tab-executive",    style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
                     dcc.Tab(label="📉 Churn Analysis",              value="tab-churn",        style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
                     dcc.Tab(label="🔍 Churn Drivers",               value="tab-drivers",      style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
                     dcc.Tab(label="⚠️  Customer Risk & Prediction", value="tab-risk",         style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
                     dcc.Tab(label="🎯 Risks, Opportunities & Actions", value="tab-actions",  style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE),
                 ],
                 style={"borderBottom": f"1px solid {CLR_BORDER}", "backgroundColor": CLR_BG}
                 ),
        style={"backgroundColor": CLR_BG}
    ),

    # Tab content
    html.Div(id="tab-content", style={"padding": "20px 24px", "backgroundColor": CLR_BG, "minHeight": "80vh"}),
])


# ── Callback ──────────────────────────────────────────────────────────────────
@app.callback(Output("tab-content", "children"), Input("main-tabs", "value"))
def render_tab(tab):
    if tab == "tab-executive":
        return tab_executive()
    elif tab == "tab-churn":
        return tab_churn_analysis()
    elif tab == "tab-drivers":
        return tab_churn_drivers()
    elif tab == "tab-risk":
        return tab_risk_prediction()
    elif tab == "tab-actions":
        return tab_risks_actions()
    return html.Div("Select a tab.")


# === ENTRY POINT ==============================================================


## Launch the Dashboard

Run the cell below to start the Dash server, then open **http://127.0.0.1:8050** in your browser.

The server runs in the notebook process. Use **Kernel → Interrupt** to stop it.

In [ ]:
if __name__ == '__main__':
    app.run(debug=False, host='127.0.0.1', port=8050)
